# P83 — Visualizar datos con t-SNE

## 1. Título y paper

**Paper:** *Visualizing Data using t-SNE*  
**Autoría:** Laurens van der Maaten, Geoffrey Hinton  
**Año y venue:** 2008 · Journal of Machine Learning Research, 9, 2579–2605  
**Nivel:** L3 · **Motor:** `tsne`  
**Ficha completa:** [`P83_tsne`](../../papers/foundational/P83_tsne/README.md)

**Hito:** Hace visibles las estructuras locales de datos de alta dimensión, y con ello se convierte en la figura por defecto de media década de artículos.

- [JMLR 9:2579–2605](https://www.jmlr.org/papers/v9/vandermaaten08a.html)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Al proyectar de muchas dimensiones a dos, los puntos moderadamente distantes se apiñan en el centro: en dimensión alta hay mucho más «sitio lejos» que cerca, y una gaussiana en el mapa no puede acomodarlo. Es el problema del apiñamiento.
2. Ejecutar una implementación mínima de la propuesta: Convertir distancias en probabilidades de vecindad, y usar en el mapa una distribución t de Student de un grado de libertad. Su cola pesada deja sitio a los puntos lejanos sin comprimir los cercanos.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Hinton y Roweis (2002), SNE
- P53
- P73


## 4. Intuición

En dimensión alta hay muchísimo más «sitio lejos» que cerca. Al aplastar a dos dimensiones, todo lo moderadamente lejano se apiña en el centro. La solución de t-SNE es usar en el mapa una distribución con cola pesada, que deja sitio a lo lejano sin comprimir lo cercano.


## 5. Concepto mínimo

```text
En el espacio original:  p_ij  ∝ exp(−‖xᵢ − xⱼ‖² / 2σ²)      gaussiana
En el mapa:              q_ij  ∝ (1 + ‖yᵢ − yⱼ‖²)⁻¹           t de Student, 1 g.l.

Minimizar  KL(P ‖ Q)  por descenso de gradiente

La cola de Student deja MUCHÍSIMA más masa lejos: eso resuelve el apiñamiento.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('tsne', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Conservarán dos ejecuciones distintas los mismos vecinos?
2. ¿Colocarán los puntos en los mismos sitios?
3. ¿Cuánta más masa deja la t de Student a distancia 8?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('tsne', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('tsne', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Las dos ejecuciones conservan la misma proporción de vecinos (**0,9556** ambas) y colocan los puntos en sitios distintos: el desplazamiento medio es **2,85**. A distancia 8, la t de Student deja una masa del orden de **10¹²** veces mayor que la gaussiana.


## 10. Comentario pedagógico

De ahí las tres reglas de lectura que casi nadie aplica: la posición absoluta no significa nada, la distancia entre grupos no es interpretable y el tamaño aparente de un grupo tampoco. Lo único que t-SNE promete preservar es la **vecindad**. Es una herramienta de exploración, no una reducción de dimensionalidad para alimentar otro modelo.


## 11. Error o anti-patrón deliberado

Anti-patrón: interpretar la distancia entre dos grupos en un mapa t-SNE.


In [ ]:
print('«Estos dos grupos estan lejos, luego son muy distintos»: no se sigue.')
print('t-SNE optimiza vecindades locales; las distancias grandes no estan restringidas.')
print('Y con otra semilla, los mismos grupos pueden quedar mas cerca o mas lejos.')

## 12. Corrección

Lo que sí se puede afirmar de un mapa t-SNE:


In [ ]:
r = run_paper_lab('tsne', seed=7)['result']
print('vecinos conservados, ejecucion 1:', r['vecinos_conservados_ejecucion_1'])
print('vecinos conservados, ejecucion 2:', r['vecinos_conservados_ejecucion_2'])
print('desplazamiento medio entre ambas:', r['desplazamiento_medio_entre_ejecuciones'])
print('-> la VECINDAD es estable; la POSICION no.')

## 13. Desafío guiado

Mira la tabla de colas y calcula a partir de qué distancia la diferencia entre gaussiana y Student pasa de ser un factor pequeño a varios órdenes de magnitud.


In [ ]:
r = run_paper_lab('tsne', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica t-SNE a un conjunto real con dos perplejidades muy distintas y dos semillas. Documenta qué conclusiones sobreviven a los cuatro mapas y cuáles no.


## 15. Evidencia de aprendizaje

Guarda la comparación entre las dos ejecuciones y tus tres reglas de lectura de un mapa t-SNE.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P83_tsne/README.md) · evaluación formal: [`assessments/papers/P83_tsne.md`](../../assessments/papers/P83_tsne.md)


## 16. Cierre

Ya se ve la estructura. Ahora lo contrario: encontrar los puntos que no pertenecen a ninguna.


## 17. Conexión con el siguiente hito

- P05
- P18

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
